In [1]:
import pandas as pd
import numpy as np

path = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/"
train = pd.read_parquet("/kaggle/input/notebooks/fatimawaseem1111/parquet/train.parquet")
articles = pd.read_csv(path + "articles.csv", dtype={"article_id": str})

item_list = sorted(train["article_id"].unique())
items = articles.set_index("article_id").loc[item_list].reset_index()
print(len(items), "items")

28593 items


In [2]:
items["text"] = items["detail_desc"].fillna(items["prod_name"])
print(items["text"].isnull().sum(), "missing")
print(items["text"].head())

0 missing
0              Jersey top with narrow shoulder straps.
1              Jersey top with narrow shoulder straps.
2    Microfibre T-shirt bra with underwired, moulde...
3    Semi shiny nylon stockings with a wide, reinfo...
4    Tights with built-in support to lift the botto...
Name: text, dtype: object


In [3]:
!pip install -q sentence-transformers

In [4]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
text_emb = text_model.encode(items["text"].tolist(), batch_size=256,
                             show_progress_bar=True, normalize_embeddings=True)
print(text_emb.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

(28593, 384)


In [5]:
np.save("/kaggle/working/text_emb.npy", text_emb)
pd.Series(item_list).to_csv("/kaggle/working/item_list.csv", index=False)

In [6]:
import os
img_dir = path + "images/"
items["img_path"] = items["article_id"].apply(lambda a: f"{img_dir}{a[:3]}/{a}.jpg")
items["has_img"] = items["img_path"].apply(os.path.exists)
print(items["has_img"].value_counts())

has_img
True     28556
False       37
Name: count, dtype: int64


In [7]:
import torch, torchvision
from torchvision import transforms
from PIL import Image

resnet = torchvision.models.resnet50(weights="IMAGENET1K_V2")
resnet.fc = torch.nn.Identity()
resnet = resnet.eval().to("cuda")

prep = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

 21%|██        | 20.5M/97.8M [00:00<00:00, 215MB/s]

 46%|████▌     | 44.8M/97.8M [00:00<00:00, 237MB/s]

 70%|███████   | 68.9M/97.8M [00:00<00:00, 244MB/s]

 96%|█████████▋| 94.1M/97.8M [00:00<00:00, 252MB/s]

100%|██████████| 97.8M/97.8M [00:00<00:00, 246MB/s]

In [8]:
img_emb = np.zeros((len(items), 2048), dtype=np.float32)
batch, idxs = [], []

with torch.no_grad():
    for i, row in enumerate(items.itertuples()):
        if row.has_img:
            batch.append(prep(Image.open(row.img_path).convert("RGB")))
            idxs.append(i)
        if len(batch) == 128 or (i == len(items) - 1 and batch):
            out = resnet(torch.stack(batch).to("cuda")).cpu().numpy()
            img_emb[idxs] = out
            batch, idxs = [], []
        if i % 5000 == 0:
            print(i, flush=True)

img_emb = img_emb / (np.linalg.norm(img_emb, axis=1, keepdims=True) + 1e-8)
print(img_emb.shape)

0


5000


10000


15000


20000


25000


(28593, 2048)


In [9]:
np.save("/kaggle/working/img_emb.npy", img_emb)

In [10]:
from transformers import CLIPModel, CLIPProcessor

fclip = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").eval().to("cuda")
fproc = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [11]:
def fclip_image_emb(inputs):
    out = fclip.vision_model(pixel_values=inputs["pixel_values"])
    return fclip.visual_projection(out.pooler_output)

def fclip_text_emb(inputs):
    out = fclip.text_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
    return fclip.text_projection(out.pooler_output)

In [12]:
fclip_img = np.zeros((len(items), 512), dtype=np.float32)
batch, idxs = [], []

with torch.no_grad():
    for i, row in enumerate(items.itertuples()):
        if row.has_img:
            batch.append(Image.open(row.img_path).convert("RGB"))
            idxs.append(i)
        if len(batch) == 128 or (i == len(items) - 1 and batch):
            inputs = fproc(images=batch, return_tensors="pt").to("cuda")
            fclip_img[idxs] = fclip_image_emb(inputs).cpu().numpy()
            batch, idxs = [], []
        if i % 5000 == 0:
            print(i, flush=True)

fclip_img = fclip_img / (np.linalg.norm(fclip_img, axis=1, keepdims=True) + 1e-8)
print(fclip_img.shape)

0


5000


10000


15000


20000


25000


(28593, 512)


In [ ]:
texts = items["text"].tolist()
fclip_txt = []
with torch.no_grad():
    for s in range(0, len(texts), 256):
        inputs = fproc(text=texts[s:s+256], return_tensors="pt",
                       padding=True, truncation=True, max_length=77).to("cuda")
        fclip_txt.append(fclip_text_emb(inputs).cpu().numpy())
fclip_txt = np.vstack(fclip_txt)
fclip_txt = fclip_txt / (np.linalg.norm(fclip_txt, axis=1, keepdims=True) + 1e-8)
print(fclip_txt.shape)

In [ ]:
np.save("/kaggle/working/fclip_img.npy", fclip_img)
np.save("/kaggle/working/fclip_txt.npy", fclip_txt)